# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(metadata.name + ': ' + metadata.description)

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore record sets, fields, and column IDs
record_sets = []

# The Croissant schema exposes record sets via metadata.record_sets
for rs in metadata.record_sets:
    print(f"RecordSet: {rs['@id']}")
    record_sets.append(rs['@id'])
    print(f"  Name: {rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    elif fields is None:
        fields = []
    for field in fields:
        print(f"    Field: {field['@id']}  (Name: {field.get('name', 'N/A')})")
        columns = field.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        elif columns is None:
            columns = []
        for col in columns:
            print(f"      Column: {col['@id']}  (Name: {col.get('name', 'N/A')})")
    print('---')

# Display the first few records for the first available record set (if any)
if record_sets:
    print(f"\nSample records from first RecordSet @{record_sets[0]}")
    for i, x in enumerate(dataset.records(record_set=record_sets[0])):
        print(x)
        if i >= 2:
            break
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
All entities are referenced by their `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_sets:
    # Load all records from this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded RecordSet '{record_set_id}' with {len(df)} records and {len(df.columns)} columns.")
    print(f"Columns (@id): {df.columns.tolist()}")

# Preview first few rows for first record set
if record_sets:
    first_rs = record_sets[0]
    print(f"\nFirst few records from RecordSet '{first_rs}':")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps such as filtering records, normalizing fields, and grouping data.

All references use `@id`.

In [ ]:
# Choose a record set and fields for EDA

# For demonstration: Use first record set and choose likely numeric fields
rs_id = record_sets[0] if record_sets else None

if rs_id:
    df = dataframes[rs_id]

    # Print all column '@id's and their types
    print("Column @ids:")
    print(df.dtypes)

    # Try to find numeric fields
    numeric_field_id = None
    group_field_id = None

    # Use dtypes to select int/float columns
    num_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    cat_cols = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]

    if num_cols:
        numeric_field_id = num_cols[0]
        print(f"Selected numeric field: {numeric_field_id}")
    if cat_cols:
        group_field_id = cat_cols[0] # e.g., a categorical field
        print(f"Selected group field: {group_field_id}")

    # Filtering
    if numeric_field_id:
        # Use quantile to set threshold dynamically
        threshold = df[numeric_field_id].quantile(0.75) if not df[numeric_field_id].isnull().all() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} (75th percentile):")
        print(filtered_df.head())

        # Normalization
        if len(filtered_df) > 0:
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized values for {numeric_field_id}:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Group and aggregate
            if group_field_id and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Mean {numeric_field_id} by {group_field_id}:")
                print(grouped_df.head())
    else:
        print("No numeric field available for analysis.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields.

In [ ]:
# Visualization examples
if rs_id and numeric_field_id:
    df = dataframes[rs_id]
    plt.figure(figsize=(8,5))
    plt.hist(df[numeric_field_id].dropna(), bins=10, color='dodgerblue', alpha=0.7)
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {rs_id}")
    plt.show()

    # If group_field_id available, use box plots
    if group_field_id:
        plt.figure(figsize=(8,5))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No appropriate numeric or group field for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded the FAIR^2 dataset and displayed metadata.
- Enumerated and extracted available record sets and fields, referencing entities by their `@id`.
- Performed basic filtering and normalization on a numeric field, and grouped by a categorical field.
- Visualized numeric field distributions and categorical group differences.

**Next steps:**
- Explore relationships between additional fields and clinical variables.
- Apply more advanced statistical or ML methods for clinicopathological analysis.